# E4 · Inyección-recuperación

**Spec:** [`docs/spec_E4_codex_injection_recovery.md`](../docs/spec_E4_codex_injection_recovery.md)  |  **Bloque:** E · Resultado  |  **Run por defecto:** `ROXs12b_realigned`

Inyecta señal sintética y mide el throughput de cada método (grid completo 112 casos).

| | |
|---|---|
| **Entrada** | Extractores reales (C2/C3/C4) + PSF |
| **Salida (QC/productos)** | `stages/stage_h04_qc.json`, `tables/injection_throughput_by_method.csv` |
| **Consume aguas abajo** | E3 (throughput), G1 |


## Qué hace E4 y qué valida

E4 **inyecta** señales sintéticas de Hα de flujo/SNR conocidos en la posición del compañero y mide cuánto **recupera** cada método → el **throughput** (flujo recuperado / inyectado). Ese factor es el que **E3 usa** para corregir el límite de flujo.

**Grid:** 112 inyecciones × 4 métodos (+ perturbaciones de PSF).

**Throughput @ SNR5:** psffit **0.667**, optimal_psfsub **0.667** (recuperan ~2/3), aperture **0.375**, optimal_ls **0.194** (insensibles — el compañero está en el gradiente del halo, en el borde del campo).

**Verificaciones:**
- **v2_nulls_clean PASS** (64 nulos, 0 hits → sin falsos positivos).
- **v3_monotonic PASS**, **v5_continuum PASS** (0% degradación).
- **v4_hierarchy FAIL** (`optimal_ls` rompe el orden esperado de throughput) — es la **patología de borde** (ls/aperture insensibles en el gradiente del halo); el canónico **psffit no se ve afectado** → limitación aceptada en F1.
- **v1_regression `unavailable`** (la regresión histórica de Stage06 no está disponible).

**Salvedad** (open_issue): la regresión histórica de Stage06 no pasó → 'H04 no válido para E3' formalmente; los throughputs se usan con esa salvedad declarada.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h04_injection.sh --run-id $RUN
```

Pesado (~22 min grid 112 casos con `h04_process_pool`).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h04_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h04_injection.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h04_qc.json', RUN_ID)
nb.show(qc, keys=['per_method_at_snr5', 'v2_nulls_clean.status', 'v4_hierarchy.status', 'n_injections'], title='E4')


## Resultados que llevaron a la conclusión

Throughput por método y las 5 verificaciones del `stage_h04_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E4', 'stages/stage_h04_qc.json'):
        q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID)
        th = q['throughput']['per_method_at_snr5']; ck = q['checks']
        print(f"grid: {q['grid']['n_injections']} inyecciones × {len(q['grid']['methods'])} métodos")
        print('\nthroughput @ SNR5:')
        for m in ['psffit','optimal_psfsub','aperture','optimal_ls']:
            print(f"   {m:15s} {th[m]['throughput']:.3f} ± {th[m]['err']:.3f}")
        print('\nverificaciones:')
        for name, c in ck.items():
            extra = f"  {c.get('failures')}" if c.get('failures') else ''
            print(f"   {name:16s} {c['status']}{extra}")
        print('\nopen_issues:')
        for s in q['open_issues']:
            print('  -', s)


## Plot 1 — throughput por método (lo que E3 consume)

El throughput @ SNR5 por método. **psffit/psfsub ~0.67** (verde); aperture 0.37 y **optimal_ls 0.19** (rojo) son insensibles en el borde → `v4_hierarchy` falla (ls rompe el orden). El canónico psffit no se ve afectado.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID)
    th = q['throughput']['per_method_at_snr5']; v4 = q['checks']['v4_hierarchy']
    methods = ['psffit', 'optimal_psfsub', 'aperture', 'optimal_ls']
    vals = [th[m]['throughput'] for m in methods]; errs = [th[m]['err'] for m in methods]
    cols = ['tab:green', 'tab:green', 'tab:orange', 'tab:red']
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(range(len(methods)), vals, yerr=errs, color=cols, capsize=4)
    for i, v in enumerate(vals): ax.text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=9)
    ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('throughput @ SNR5 (recuperado/inyectado)'); ax.set_ylim(0, 0.9)
    ax.set_title(f"E4 · throughput por método (v4_hierarchy={v4['status']}: {v4['failures']} rompe el orden)")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'e4_injection'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'throughput.png', dpi=110); print('figura ->', outdir / 'throughput.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la curva de recuperación (pendiente = throughput)

Flujo neto recuperado vs inyectado por método. **Subconjunto mostrado:** solo la variante `nominal` con `continuum_mode='none'` (la configuración base del grid; las demás variantes/knobs se resumen en el QC). Pasa por el origen y la **pendiente es el throughput**: psffit/psfsub siguen ~0.67 (paralelos, bajo el 1:1 ideal); aperture 0.37 y ls 0.19 son mucho más planos (insensibles). *(psffit y psfsub coinciden en 0.67; psffit va discontinuo para verse.)*


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID); th = q['throughput']['per_method_at_snr5']
    d = pd.read_csv(rd / 'tables' / 'injection_throughput_by_method.csv')
    d = d[(d.variant == 'nominal') & (d.continuum_mode == 'none')]
    styles = {'psffit': ('tab:green', '--'), 'optimal_psfsub': ('tab:blue', '-'),
              'aperture': ('tab:orange', '-'), 'optimal_ls': ('tab:red', '-')}
    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    for m, (c, ls) in styles.items():
        g = d[d.method == m].groupby('injected_flux')['recovered_flux_net'].median()
        ax.plot(g.index, g.values, 'o' + ls, color=c, ms=5, label=f"{m} (T={th[m]['throughput']:.2f})")
    mx = d['injected_flux'].max(); ax.plot([0, mx], [0, mx], 'k:', lw=0.8, label='ideal 1:1 (T=1)')
    ax.set_xlabel('flujo inyectado'); ax.set_ylabel('flujo neto recuperado (mediana)')
    ax.set_title('E4 · recuperación: pendiente = throughput; aperture/ls insensibles en el borde')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'e4_injection'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'recovery.png', dpi=110); print('figura ->', outdir / 'recovery.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Throughput @SNR5: psffit/psfsub ≈ 0.67** (canónico psffit); aperture 0.37, optimal_ls 0.19 (insensibles en el borde del halo). Son los factores que E3 usa.
- **v2_nulls/v3_monotonic/v5 PASS**; **v4_hierarchy FAIL** (optimal_ls rompe el orden por la patología de borde) — canónico psffit NO afectado → limitación aceptada en F1.
- Salvedad: la regresión histórica de Stage06 no pasó ('H04 no válido para E3' formalmente); throughputs usados con la salvedad declarada.


## Conclusión (registrada)

**E4: throughput @SNR5 medido — psffit/psfsub 0.67, aperture 0.37, optimal_ls 0.19.**

- **Fecha:** grid completo de 112 casos, 2026-07-08.
- **Valida** los throughputs que E3 consume; recuperación limpia en los nulos (v2 PASS, 0 hits).
- **v4_hierarchy FAIL** por optimal_ls (patología de borde: ls/aperture insensibles en el gradiente del halo); el canónico psffit no se afecta → limitación aceptada.
- **v1_regression unavailable**; open_issue: la regresión histórica de Stage06 no pasó ('H04 no válido para E3' formalmente).
- **Downstream:** el throughput psffit 0.67 es el que E3 aplica para el límite de Ṁ.
